<a href="https://colab.research.google.com/github/romeurf/DipRadar/blob/main/ml_training/DipRadar_Training_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 DipRadar — ML Training (Colab)

Pipeline completo de treino dos modelos ML que correm no Railway.

**Outputs gerados e pushed para GitHub:**
- `ml_training/dip_models.pkl` → bundle do modelo (joblib)
- `ml_training/ml_report.json` → métricas e metadados do treino
- `ml_training_base.parquet` → dataset de treino completo

**Fluxo (correr em ordem):**
1. Célula 1 — Secrets + clone do repo + install deps
2. Célula 2 — Download de preços via yfinance (ETFs + ~800 tickers do universo)
3. Célula 2.5 — Enriquecimento de sectores via yfinance (sem sectors.py)
4. Célula 3 — Gerar alertas históricos + feature defs
5. Célula 3b — Download macro histórico (^VIX, SPY, ^TNX, ^IRX, HYG, LQD, IYT, XLI)
6. Célula 4 — Build do dataset de treino via `build_dataset_v31()` → shape (N, features)
7. Célula 5 — Walk-forward CV + seleção do champion
8. Célula 6 — Treino full + calibrador isotónico
9. Célula 7 — Bundle + Report
10. Célula 8 — Guardar parquet actualizado
11. Célula 9 — Push automático para GitHub (Railway redeploy automático)


## ⚙️ Célula 1 — Setup: secrets, clone, install deps

In [ ]:
import os, subprocess, sys
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_API_KEY')
REPO_OWNER   = 'romeurf'
REPO_NAME    = 'DipRadar'
BRANCH       = 'main'

assert GITHUB_TOKEN, 'Falta GITHUB_TOKEN nos secrets do Colab'
print('✅ Secrets carregados')

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth=1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✅ Repo em {REPO_DIR}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'joblib', 'scikit-learn', 'lightgbm', 'xgboost',
                'pandas', 'pyarrow', 'yfinance', 'imbalanced-learn'], check=True)
print('✅ Dependências instaladas')

## 🌐 Célula 2 — Download de preços via yfinance (~800 tickers)

In [ ]:
import yfinance as yf
import pandas as pd
from pathlib import Path
from ml_training.config import DEFAULT_ETF, SECTOR_ETF
from ml_training.price_fetch import load_etf_cache

REPO_PATH     = Path(REPO_DIR)
ETF_CACHE_DIR = REPO_PATH / 'ml_training' / 'etf_cache'
ETF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

etf_tickers = sorted(set([DEFAULT_ETF] + list(SECTOR_ETF.values())))
print(f'ETFs a descarregar: {etf_tickers}')

etf_cache = load_etf_cache(
    etfs=etf_tickers,
    start='2018-01-01',
    cache_dir=ETF_CACHE_DIR,
)

print(f'\n✅ ETF cache: {len(etf_cache)}/{len(etf_tickers)} ETFs carregados')
if len(etf_cache) == 0:
    raise RuntimeError('Nenhum ETF carregado — verifica a ligação à Internet no Colab.')

In [ ]:
try:
    from universe import (
        _SP500_FALLBACK, _NASDAQ100_FALLBACK, _STOXX200, _FTSE100,
        USER_PORTFOLIO, USER_WATCHLIST, ETF_TICKERS
    )
    _raw = USER_PORTFOLIO + USER_WATCHLIST + _SP500_FALLBACK + _NASDAQ100_FALLBACK + _STOXX200 + _FTSE100
    _seen = set()
    all_tickers = []
    for t in _raw:
        t = t.strip().upper()
        if t and t not in _seen and t not in ETF_TICKERS:
            _seen.add(t)
            all_tickers.append(t)
    all_tickers = sorted(all_tickers)
    print(f'✅ universe.py: {len(all_tickers)} tickers')
except Exception as e:
    print(f'⚠️  universe.py falhou ({e}) — lista mínima')
    ETF_TICKERS = set()
    all_tickers = sorted(['AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA','BRK-B',
        'JNJ','UNH','LLY','ABBV','MRK','ABT','TMO','DHR',
        'JPM','BAC','WFC','GS','MS','BLK',
        'PG','KO','PEP','WMT','COST','MCD','HD','TGT',
        'XOM','CVX','COP','SLB','NEE','DUK','SO',
        'CAT','DE','HON','MMM','GE','RTX','VZ','T','CMCSA',
        'AMT','PLD','EQIX','LIN','APD','SHW',
        'NVO','ADBE','UBER','ADP','CRM','CRWD','PLTR','NOW',])

In [ ]:
from ml_training.price_fetch import fetch_ohlcv_batch
from datetime import date

START_DATE = '2018-01-01'
END_DATE   = date.today().strftime('%Y-%m-%d')

print(f'A descarregar preços de {len(all_tickers)} tickers...')
price_cache = fetch_ohlcv_batch(
    tickers_list=all_tickers,
    start=START_DATE,
    end=END_DATE,
    batch_size=40,
    progress_log=True,
)
not_found = [t for t in all_tickers if t not in price_cache]
print(f'\n✅ Price cache: {len(price_cache)}/{len(all_tickers)} tickers')
print(f'   Não encontrados: {len(not_found)}')

## 🗂️ Célula 2.5 — Enriquecimento de sectores via yfinance

In [ ]:
import time

try:
    from universe import TICKER_SECTOR as _STATIC_SECTOR_MAP
    print(f'✅ universe.TICKER_SECTOR: {len(_STATIC_SECTOR_MAP)} entradas estáticas')
except ImportError:
    _STATIC_SECTOR_MAP = {}
    print('ℹ️  TICKER_SECTOR não existe — sector 100% via yfinance')

sector_map: dict[str, str] = {}
missing_sector = [t for t in price_cache if t not in _STATIC_SECTOR_MAP]
print(f'\nA buscar sector para {len(missing_sector)} tickers...')

found = 0; errors = 0
for i, ticker in enumerate(missing_sector):
    try:
        info_full = yf.Ticker(ticker).info
        sec = info_full.get('sector') or info_full.get('sectorKey') or ''
        sector_map[ticker] = sec if sec else 'Unknown'
        if sec: found += 1
    except Exception:
        sector_map[ticker] = 'Unknown'; errors += 1
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(missing_sector)} | ok={found} erros={errors}')
        time.sleep(0.3)

full_sector_map: dict[str, str] = {**_STATIC_SECTOR_MAP, **sector_map}
from collections import Counter
dist = Counter(v for v in full_sector_map.values() if v and v != 'Unknown')
print(f'\n✅ Sector map: {len(full_sector_map)} tickers, {sum(dist.values())} com sector')

def get_ticker_sector(ticker: str) -> str:
    return full_sector_map.get(ticker, 'Unknown') or 'Unknown'

print('✅ get_ticker_sector() disponível')

## 🏗️ Célula 3 — Gerar alertas históricos + feature defs

In [ ]:
import numpy as np
import math

from ml_training.config import (
    DEFAULT_ETF, SECTOR_ETF, HORIZON_DAYS,
    WINSOR_ABS_LO, WINSOR_ABS_HI,
)
from ml_features import (
    FEATURE_COLUMNS, _FALLBACK, add_derived_features, add_momentum_features
)

def _rsi(closes: pd.Series, period: int = 14) -> float:
    if len(closes) < period + 1:
        return float(_FALLBACK.get('rsi_14', 50.0))
    delta = closes.diff().dropna()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    rsi_val = (100 - 100 / (1 + rs)).iloc[-1]
    return float(rsi_val) if pd.notna(rsi_val) else float(_FALLBACK.get('rsi_14', 50.0))

def _atr_ratio(hist: pd.DataFrame, period: int = 14) -> float:
    if len(hist) < period + 1:
        return float(_FALLBACK.get('atr_ratio', 0.02))
    tr = pd.concat([
        hist['High'] - hist['Low'],
        (hist['High'] - hist['Close'].shift()).abs(),
        (hist['Low'] - hist['Close'].shift()).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(period).mean().iloc[-1]
    close = hist['Close'].iloc[-1]
    return float(atr / close) if close > 0 else float(_FALLBACK.get('atr_ratio', 0.02))

def _volume_spike(hist: pd.DataFrame, window: int = 20) -> float:
    if len(hist) < window:
        return 1.0
    avg = hist['Volume'].iloc[-window:].mean()
    last = hist['Volume'].iloc[-1]
    return float(last / avg) if avg > 0 else 1.0

def build_v2_features(hist: pd.DataFrame, alert_date: pd.Timestamp) -> dict:
    h = hist[hist.index <= alert_date]
    if h.empty: return {}
    close = h['Close'].iloc[-1]
    high_52w = h['High'].iloc[-252:].max() if len(h) >= 5 else close
    prev_close = h['Close'].iloc[-2] if len(h) >= 2 else close
    drop_today = (close / prev_close - 1) if prev_close > 0 else 0.0
    drawdown_52w = (close / high_52w - 1) if high_52w > 0 else 0.0
    return {
        'drop_pct_today': float(drop_today),
        'drawdown_52w': float(drawdown_52w),
        'rsi_14': _rsi(h['Close']),
        'atr_ratio': _atr_ratio(h),
        'volume_spike': _volume_spike(h),
    }

def build_targets_close(alert_date: pd.Timestamp, hist: pd.DataFrame,
                        horizon: int = HORIZON_DAYS) -> dict:
    entry_slice = hist[hist.index <= alert_date]
    if entry_slice.empty:
        return {'close_60d': float('nan'), 'max_drawdown_60d': float('nan')}
    entry_price = float(entry_slice['Close'].iloc[-1])
    if entry_price <= 0:
        return {'close_60d': float('nan'), 'max_drawdown_60d': float('nan')}
    fwd = hist[
        (hist.index > alert_date) &
        (hist.index <= alert_date + pd.Timedelta(days=horizon))
    ]
    if len(fwd) < 5:
        return {'close_60d': float('nan'), 'max_drawdown_60d': float('nan')}
    close_ret = float(fwd['Close'].iloc[-1] / entry_price - 1)
    max_draw = float(fwd['Low'].min() / entry_price - 1)
    return {'close_60d': close_ret, 'max_drawdown_60d': max_draw}

def spy_close_return_forward(spy_hist, alert_date, horizon=HORIZON_DAYS):
    if spy_hist is None: return float('nan')
    entry_slice = spy_hist[spy_hist.index <= alert_date]
    if entry_slice.empty: return float('nan')
    spy_entry = float(entry_slice['Close'].iloc[-1])
    if spy_entry <= 0: return float('nan')
    fwd = spy_hist[
        (spy_hist.index > alert_date) &
        (spy_hist.index <= alert_date + pd.Timedelta(days=horizon))
    ]
    if len(fwd) < 5: return float('nan')
    return float(fwd['Close'].iloc[-1] / spy_entry - 1)

def days_since_52w_high(hist: pd.DataFrame, alert_date: pd.Timestamp) -> float:
    window = hist[
        (hist.index <= alert_date) &
        (hist.index > alert_date - pd.Timedelta(days=365))
    ]
    if len(window) < 20: return 60.0
    high_idx = window['High'].idxmax()
    return float((alert_date - high_idx).days)

print('✅ Funções auxiliares definidas (target: close_60d)')

In [ ]:
from ml_training.models import build_feature_lists

FEATURE_COLS, FEATURE_COLS_BASELINE = build_feature_lists()

print(f'Features totais  : {len(FEATURE_COLS)}')
print(f'Features baseline: {len(FEATURE_COLS_BASELINE)}')
print(f'Target           : close_60d (close-to-close) vs spy_close_60d')
print(f'vix_regime       : 3 bins — 0=low(vix<15), 1=normal(15-25), 2=alta vol(vix>25)')

In [ ]:
from ml_training.config import SUBSAMPLE_YEARS, MAX_ALERTS_PER_YEAR, SUBSAMPLE_SEED
from ml_training.data import generate_historical_alerts

base_df = generate_historical_alerts(
    all_tickers=all_tickers,
    price_cache=price_cache,
    sector_fn=get_ticker_sector,
    subsample_years=SUBSAMPLE_YEARS,
    max_per_year=MAX_ALERTS_PER_YEAR,
    seed=SUBSAMPLE_SEED,
)

print(f'✅ Alertas gerados: {len(base_df)}')
if not base_df.empty:
    print(f'   Período: {base_df["alert_date"].min().date()} → {base_df["alert_date"].max().date()}')
    print(f'   Tickers únicos: {base_df["ticker"].nunique()}')

## 📊 Célula 3b — Download macro histórico (uma vez)

In [ ]:
from ml_training.data import MACRO_TICKERS
from ml_training.price_fetch import fetch_ohlcv_batch

print(f'A descarregar macro tickers: {MACRO_TICKERS}')
macro_price_cache = fetch_ohlcv_batch(
    tickers_list=MACRO_TICKERS,
    start='2018-01-01',
    end=END_DATE,
    batch_size=10,
    progress_log=False,
)
print(f'✅ Macro cache: {len(macro_price_cache)}/{len(MACRO_TICKERS)} tickers')

## 🏗️ Célula 4 — Build do dataset de treino


In [ ]:
from ml_training.data import build_dataset_v31

df_train, skipped = build_dataset_v31(
    base_df=base_df,
    price_cache=price_cache,
    etf_cache=etf_cache,
    feature_cols_v31=FEATURE_COLS,
    macro_price_cache=macro_price_cache,
)

print(f'\n✅ Dataset construído: {df_train.shape}')
print(f'   Skipped: {skipped}')
if not df_train.empty:
    print(f'   Período: {df_train["alert_date"].min().date()} → {df_train["alert_date"].max().date()}')
    missing_cols = [c for c in FEATURE_COLS if c not in df_train.columns]
    print(f'\n   Features esperadas: {len(FEATURE_COLS)}')
    if missing_cols:
        print(f'   ⚠️  Colunas em falta no df: {missing_cols}')
    else:
        print(f'   ✅ Todas as {len(FEATURE_COLS)} features presentes no df')

## 📊 Distribuição de alpha_60d e análise de features

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(df_train['alpha_60d'], bins=80, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].axvline(0,    color='red',   linewidth=1.5, linestyle='--', label='Benchmark (SPY)')
axes[0].axvline(0.05, color='green', linewidth=1.5, linestyle='--', label='Target >5%')
axes[0].set_title('Distribuição alpha_60d (winsorized)')
axes[0].set_xlabel('alpha_60d')
axes[0].legend()

yr_counts = df_train.groupby(df_train['alert_date'].dt.year).size()
axes[1].bar(yr_counts.index, yr_counts.values, color='steelblue', alpha=0.8)
axes[1].set_title('Alertas por ano (pós subsample)')
axes[1].set_xlabel('Ano')

if 'vix_regime' in df_train.columns:
    vc     = df_train['vix_regime'].value_counts().sort_index()
    labels = ['Low (vix<15)', 'Normal (15-25)', 'Alta Vol (vix>25)']
    vals   = [vc.get(0.0, 0), vc.get(1.0, 0), vc.get(2.0, 0)]
    colors = ['steelblue', 'orange', 'tomato']
    axes[2].bar(labels, vals, color=colors, alpha=0.8)
    axes[2].set_title('vix_regime distribution')
else:
    axes[2].set_title('vix_regime: N/A')

plt.tight_layout()
plt.savefig('alpha_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot guardado: alpha_distribution.png')

## 🔄 Célula 5 — Walk-forward CV (10 folds)

In [ ]:
from ml_training.train import run_walk_forward_cv, summarize_results, select_champion
from ml_training.config import N_FOLDS, PURGE_DAYS
from ml_training.models import build_model_configs

MODEL_CONFIGS = build_model_configs(
    feature_cols_v31=FEATURE_COLS,
    feature_cols_baseline=FEATURE_COLS_BASELINE,
)

print(f'Modelos candidatos: {list(MODEL_CONFIGS.keys())}')
print(f'Folds: {N_FOLDS}  |  Purge: {PURGE_DAYS}d')

cv_results, oof_pred, fold_specs = run_walk_forward_cv(
    df_v31=df_train,
    model_configs=MODEL_CONFIGS,
    n_folds=N_FOLDS,
    purge_days=PURGE_DAYS,
)

print('\n✅ CV completo')

In [ ]:
import pandas as pd

summary = summarize_results(cv_results)
print('CV Results por modelo:')
print(summary[['model','rho_alpha_mean','rho_alpha_std','rho_down_mean','topk_pnl_mean','n_folds']].to_string(index=False))

CHAMPION_NAME, champion_row = select_champion(summary)
CHAMPION_CFG  = MODEL_CONFIGS[CHAMPION_NAME]
print(f'\n🏆 Champion: {CHAMPION_NAME}  (rho_alpha={champion_row["rho_alpha_mean"]:.4f})')

In [ ]:
import matplotlib.pyplot as plt

rows = []
for model_name, folds in cv_results.items():
    for f in folds:
        rows.append({'model': model_name, **f})
cv_df = pd.DataFrame(rows)

pivot = cv_df.pivot_table(index='fold', columns='model', values='rho_alpha')
pivot.plot(figsize=(12, 5), marker='o', title='rho_alpha por fold — Walk-forward CV')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.ylabel('Spearman rho (alpha_60d)')
plt.tight_layout()
plt.savefig('cv_rho_alpha.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot guardado: cv_rho_alpha.png')

## 🎓 Célula 6 — Treino full + calibrador isotónico

In [ ]:
from ml_training.train import train_full_champion, fit_isotonic_calibrator

champ_alpha, champ_down, feats_used, n_train = train_full_champion(
    df_v31=df_train,
    champion_cfg=CHAMPION_CFG,
)

isotonic_cal, brier_oof, n_oof = fit_isotonic_calibrator(
    oof_pred_champion=oof_pred[CHAMPION_NAME],
    alpha_true=df_train['alpha_60d'].values,
)

print(f'✅ Modelo full treinado: {CHAMPION_NAME}')
print(f'   Features usadas: {len(feats_used)}')
print(f'   Amostras treino: {n_train}')
print(f'   Brier OOF: {brier_oof:.4f}  (OOF samples: {n_oof})')

## 📦 Célula 7 — Bundle + Report

In [ ]:
from ml_training.bundle import DipModelsV3, save_bundle, build_report, save_report
from ml_training.config import HORIZON_DAYS, N_FOLDS, PURGE_DAYS, NEW_FEATURES_V31, NEW_FEATURES_V32
from datetime import datetime
import numpy as np

# win_rate_alpha: fracção de OOF preds onde alpha_60d real > 5%
oof_valid = oof_pred[CHAMPION_NAME]
mask = np.isfinite(oof_valid)
y_alpha = df_train['alpha_60d'].values
win_rate_alpha = float((y_alpha[mask] > 0.05).mean()) if mask.sum() > 0 else 0.0

bundle = DipModelsV3(
    model_up=champ_alpha,
    model_down=champ_down,
    feature_cols=feats_used,
    score_calibrator=isotonic_cal,
    n_train_samples=n_train,
    train_date=datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    champion_name=CHAMPION_NAME,
    schema_version=3,
    rho_alpha=float(champion_row['rho_alpha_mean']),
    rho_down=float(champion_row['rho_down_mean']),
    topk_pnl=float(champion_row['topk_pnl_mean']),
    fold_metrics=[],
)

report = build_report(
    bundle=bundle,
    summary_df=summary,
    brier_oof=brier_oof,
    win_rate_alpha=win_rate_alpha,
    n_folds_used=N_FOLDS,
    purge_days=PURGE_DAYS,
    horizon_days=HORIZON_DAYS,
    new_features=NEW_FEATURES_V31 + NEW_FEATURES_V32,
)

BUNDLE_PATH = REPO_PATH / 'ml_training' / 'dip_models.pkl'
REPORT_PATH = REPO_PATH / 'ml_training' / 'ml_report.json'

save_bundle(bundle, BUNDLE_PATH)
save_report(report, REPORT_PATH)

print(f'✅ Bundle: {BUNDLE_PATH}')
print(f'✅ Report: {REPORT_PATH}')
print(f'   Champion: {CHAMPION_NAME}')
print(f'   Features: {len(feats_used)}')

## 💾 Célula 8 — Guardar parquet actualizado

In [ ]:
BASE_PARQUET = REPO_PATH / 'ml_training_base.parquet'
df_train.to_parquet(BASE_PARQUET, index=False)
size_mb = BASE_PARQUET.stat().st_size / 1024 / 1024
print(f'✅ Parquet guardado: {BASE_PARQUET}')
print(f'   Shape: {df_train.shape}  |  Tamanho: {size_mb:.2f} MB')

## ⬇️ Célula 8a — Download local dos artefactos (opcional)

In [ ]:
from google.colab import files
files.download(str(BUNDLE_PATH))
files.download(str(REPORT_PATH))
print('✅ Download iniciado')

## 🚀 Célula 9 — Push automático para GitHub

In [ ]:
import subprocess

subprocess.run(['git', '-C', str(REPO_PATH), 'config', 'user.email', 'colab@diparadar.ai'], check=True)
subprocess.run(['git', '-C', str(REPO_PATH), 'config', 'user.name',  'Colab Training Bot'],  check=True)

subprocess.run(['git', '-C', str(REPO_PATH), 'add',
                'ml_training/dip_models.pkl',
                'ml_training/ml_report.json',
                'ml_training_base.parquet'], check=True)

result = subprocess.run(
    ['git', '-C', str(REPO_PATH), 'diff', '--cached', '--stat'],
    capture_output=True, text=True
)
print(result.stdout)

from datetime import datetime
commit_msg = f'chore: retrain {datetime.utcnow().strftime("%Y-%m-%dT%H:%M")}Z — champion={CHAMPION_NAME}'
subprocess.run(['git', '-C', str(REPO_PATH), 'commit', '-m', commit_msg], check=True)
subprocess.run(['git', '-C', str(REPO_PATH), 'push', 'origin', BRANCH], check=True)
print(f'✅ Push completo — Railway vai fazer redeploy automático')